# Statistical convergence of the QELM ridge in `PER_P`

Issue #267, follow-up of PR #261 (`experiments/haar_readouts`). The QELM reads the charge sector of a monitored Haar brickwork from the observables of the final state: `<P_k>` and `<P_i P_j>` for `P = X, Y, Z` (108 features at `N = 8`), read by a ridge regression. PR #261 found its accuracy unstable with `PER_P`, the number of trajectories per measurement probability `p`. This notebook measures how many trajectories it takes for the accuracy to stop moving, at every `p`, averaged over independent circuits.

The experiment is `experiments/per_p_convergence/per_p_convergence.py`. For each of `N_CIRCUITS` Haar circuits (one fixed draw of gates each, the same for every `p` and every trajectory) and each `p`:

1. simulate `max(PER_P_VALUES)` trajectories of `T` rounds (one brickwork period, then a Z measurement of each system qubit with probability `p`), with `T = N`, so that `2 T >= N`;
2. for each `PER_P`, keep the first `PER_P` trajectories (nested sets), split them 80 % / 20 % stratified on the label, fit `RidgeCV` (alpha by leave-one-out) and score its test accuracy. Also kept: the training accuracy and the spectrum of `X^T X` of the training features.

**What `PER_P` is here.** The features are the exact expectation values of each trajectory state. `PER_P` counts training samples (trajectories), not shots per feature: there is no shot noise on a feature. The convergence measured here is that of the training set size, and of the test set, whose 20 % of `PER_P` sets the resolution of one accuracy.

**Saturation.** With `mu(PER_P)` the accuracy averaged over the circuits, `PER_P*` of a `p` is the smallest `PER_P` of the grid from which `|mu(PER_P) - mu(2 PER_P)| < 0.005` holds for every larger pair. The universal `PER_P*` is the largest over `p`.

In [ ]:
import os
import sys
from pathlib import Path

# Each process simulates one circuit: one thread per process.
os.environ.setdefault("OMP_NUM_THREADS", "1")

# Jupyter starts in the notebook's folder. Move to the repository root.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here or through the environment variable of the same name (the test `tests/test_per_p_convergence_notebook.py` runs a small version). `N_MAIN` is the reference size of the issue, run at every `P_VALUES`; the other sizes of `N_VALUES` are run at the `REGIMES` only (volume law `0.05`, near `p_c` `0.18`, area law `0.40`), for the projection of `PER_P*` with `N`. `N_JOBS = 0` uses every core. Each sweep goes to `outputs/per_p_convergence/<date_time>/N<N>/`; `RUN = 0` reads back the latest one.

With the defaults, `N = 8` takes about 30 minutes on 4 cores, `N = 10` about 40 minutes.

In [ ]:
N_MAIN = int(os.environ.get("N_MAIN", "8"))
N_VALUES = [int(n) for n in os.environ.get("N_VALUES", "6 8 10").split()]
P_VALUES = os.environ.get(
    "P_VALUES", "0.02 0.05 0.08 0.1 0.12 0.15 0.18 0.2 0.25 0.3 0.4"
).split()
REGIMES = os.environ.get("REGIMES", "0.05 0.18 0.4").split()
PER_P_VALUES = os.environ.get(
    "PER_P_VALUES", "500 1000 2000 4000 8000 16000 32000"
).split()
N_CIRCUITS = int(os.environ.get("N_CIRCUITS", "20"))
N_JOBS = int(os.environ.get("N_JOBS", "0"))
SEED = int(os.environ.get("SEED", "42"))
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
EXPERIMENT = "per_p_convergence"  # this notebook's own output folder

## Run the sweeps

One run of `per_p_convergence.run` per size, in this process (its circuits in `N_JOBS` worker processes), saved with `per_p_convergence.save`: `per_p_convergence.npz` (axes circuit, p, PER_P), `summary.json` and `resolved_config.json`.

In [ ]:
import importlib.util

from QELM import load_config
from QELM.utils import latest_sweep, new_sweep_dir

FOLDER = Path("experiments/per_p_convergence").resolve()
sys.path.insert(0, str(FOLDER))  # the worker processes import the script by name


def load_script(name: str):
    spec = importlib.util.spec_from_file_location(name, FOLDER / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pc = load_script("per_p_convergence")
plot = load_script("plot_per_p_convergence")


def overrides(N: int) -> list[str]:
    p_values = P_VALUES if N == N_MAIN else REGIMES
    return [
        f"N={N}",
        f"seed={SEED}",
        f"convergence.p_values=[{','.join(p_values)}]",
        f"convergence.regimes=[{','.join(REGIMES)}]",
        f"convergence.per_p_values=[{','.join(PER_P_VALUES)}]",
        f"convergence.n_circuits={N_CIRCUITS}",
        f"convergence.n_rounds={N}",  # T = N
        f"convergence.n_jobs={N_JOBS}",
    ]


if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    for N in N_VALUES:
        print(f"--- N = {N}")
        cfg = load_config(overrides(N), config_name="per_p_convergence")
        pc.save(pc.run(cfg), cfg, sweep_dir / f"N{N}")
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
folders = sorted(sweep_dir.glob("N*"), key=lambda f: int(f.name[1:]))
runs = {plot.load_run(f)["N"]: plot.load_run(f) for f in folders}
main = runs[N_MAIN]
print(f"sweep: {sweep_dir}, sizes {sorted(runs)}")

## Convergence profile

Test accuracy against `PER_P` (log2 axis) for the three regimes, mean over the circuits with a band of ±1 σ (spread between circuits); the training accuracy is dashed. The zone `PER_P >= PER_P*`, where the mean accuracy has stopped moving at every `p`, is shaded in `#fa5703`. The gap between train and test measures the overfitting of the 108 weights; it closes as `PER_P` grows.

In [ ]:
import matplotlib.pyplot as plt

plot.plot_profile(main, sweep_dir)
plt.show()
plot.plot_steps(main, sweep_dir)
plt.show()

The second figure is the step `|mu(2 PER_P) - mu(PER_P)|`, with the standard error of the paired difference between circuits (the trajectory sets are nested, so the two accuracies of a circuit share most of their data). A step below the tolerance only means something where its error bar is below it too.

## Map of the accuracy over `(PER_P, p)`

The mean accuracy over the plane. The low-`p` corner shows how far the accuracy rises when trajectories are added; above `p ≈ 0.15` it stays near chance at every `PER_P`.

In [ ]:
plot.plot_heatmap(main, sweep_dir)
plt.show()

## Conditioning of the features

Eigenvalues of `X^T X / n`, the covariance of the centered training features, and their effective rank (participation ratio `(sum λ)^2 / sum λ^2`). A plateau of the accuracy that comes with a stable effective rank is a plateau of the information in the features, not of the fit.

In [ ]:
plot.plot_spectrum(main, sweep_dir)
plt.show()

## Decision: `PER_P` for the scaling campaign

The thresholds per `p` at the reference size, the universal value, and, with several sizes, the fit `log2 PER_P* = a + b N` per regime projected to `N = 6 .. 12`. `b` is the number of doublings of `PER_P*` per added qubit; `b = 1` would be the `2**N` growth of a concentration of the features.

In [ ]:
import json

summary = main["summary"]
print(
    f"N = {N_MAIN}, {summary['n_circuits']} circuits, tolerance {summary['tolerance']}"
)
print(f"{'p':>6} {'S_R':>6} {'mu(max)':>8} {'sigma':>6} {'PER_P*':>7}")
for p, row in summary["by_p"].items():
    print(
        f"{float(p):6.3f} {row['S_R']:6.3f} {row['mu'][-1]:8.3f} "
        f"{row['sigma'][-1]:6.3f} {str(row['per_p_star']):>7}"
    )
print(f"universal PER_P* = {summary['per_p_star']}")

law = None
if len(runs) > 1:
    plot.plot_scaling(list(runs.values()), sweep_dir)
    plt.show()
    table = plot.scaling_table(list(runs.values()))
    law = plot.projection_law(table)
    print(json.dumps({"table": table, "law": law}, indent=1))
    (sweep_dir / "decision.json").write_text(
        json.dumps(
            {"N_main": N_MAIN, "summary": summary, "table": table, "law": law},
            indent=2,
        )
    )